In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
import numpy as np

# 1. オートエンコーダーの定義（前述と同じ）
class Autoencoder(nn.Module):
    def __init__(self):
        super(Autoencoder, self).__init__()
        self.encoder = nn.Sequential(nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 8))
        self.decoder = nn.Sequential(nn.Linear(8, 128), nn.ReLU(), nn.Linear(128, 784), nn.Sigmoid())

    def forward(self, x):
        return self.decoder(self.encoder(x))

# 2. データの準備：0~1を「正常」とする
transform = transforms.Compose([transforms.ToTensor(), transforms.Lambda(lambda x: x.view(-1))])
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)

# 0-1のラベルを持つデータだけを抽出
normal_indices = [i for i, (_, label) in enumerate(train_dataset) if label < 2]
train_loader = torch.utils.data.DataLoader(
    torch.utils.data.Subset(train_dataset, normal_indices),
    batch_size=64, shuffle=True
)

# 3. 学習
model = Autoencoder()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("学習開始（0-8の数字のみ）...")
for epoch in range(5):
    for data, _ in train_loader:
        output = model(data)
        loss = criterion(output, data)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1} 完了")

# 4. しきい値（Threshold）の決定
# 正常データ（0-1）を再度流して、平均的な「復元誤差」を算出する
model.eval()
errors = []
with torch.no_grad():
    for data, _ in train_loader:
        output = model(data)
        # サンプルごとのMSEを計算
        loss_per_sample = torch.mean((output - data)**2, dim=1)
        errors.extend(loss_per_sample.tolist())

# しきい値を「平均 + 標準偏差の3倍」に設定（統計的な外れ値の基準）
threshold = np.mean(errors) + 3 * np.std(errors)
print(f"\n算出されたしきい値: {threshold:.6f}")

# 5. 検証：0-1（正常）と 2-9（異常）を判定してみる
test_dataset = datasets.MNIST(root='./data', train=False, transform=transform)

def predict(img, label):
    with torch.no_grad():
        recon = model(img)
        error = criterion(recon, img).item()
        is_anomaly = error > threshold
        status = "❌ 異常(Anomaly)" if is_anomaly else "✅ 正常(Normal)"
        print(f"ラベル {label} -> 誤差: {error:.6f} | 判定: {status}")

print("\n--- 判定テスト ---")
# 正常な例（ラベル0）
img_0, _ = [d for d in test_dataset if d[1] == 0][0]
predict(img_0, 0)

# 異常な例（ラベル1~9）
img_1, _ = [d for d in test_dataset if d[1] == 1][0]
predict(img_1, 1)
img_2, _ = [d for d in test_dataset if d[1] == 2][0]
predict(img_2, 2)
img_3, _ = [d for d in test_dataset if d[1] == 3][0]
predict(img_3, 3)
img_4, _ = [d for d in test_dataset if d[1] == 4][0]
predict(img_4, 4)
img_5, _ = [d for d in test_dataset if d[1] == 5][0]
predict(img_5, 5)
img_6, _ = [d for d in test_dataset if d[1] == 6][0]
predict(img_6, 6)
img_7, _ = [d for d in test_dataset if d[1] == 7][0]
predict(img_7, 7)
img_8, _ = [d for d in test_dataset if d[1] == 8][0]
predict(img_8, 8)
img_9, _ = [d for d in test_dataset if d[1] == 9][0]
predict(img_9, 9)

100%|██████████| 9.91M/9.91M [00:00<00:00, 40.0MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.01MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 10.2MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.16MB/s]


学習開始（0-8の数字のみ）...
Epoch 1 完了
Epoch 2 完了
Epoch 3 完了
Epoch 4 完了
Epoch 5 完了

算出されたしきい値: 0.042238

--- 判定テスト ---
ラベル 0 -> 誤差: 0.018740 | 判定: ✅ 正常(Normal)
ラベル 1 -> 誤差: 0.002375 | 判定: ✅ 正常(Normal)
ラベル 2 -> 誤差: 0.066932 | 判定: ❌ 異常(Anomaly)
ラベル 3 -> 誤差: 0.068871 | 判定: ❌ 異常(Anomaly)
ラベル 4 -> 誤差: 0.056793 | 判定: ❌ 異常(Anomaly)
ラベル 5 -> 誤差: 0.097598 | 判定: ❌ 異常(Anomaly)
ラベル 6 -> 誤差: 0.052342 | 判定: ❌ 異常(Anomaly)
ラベル 7 -> 誤差: 0.056187 | 判定: ❌ 異常(Anomaly)
ラベル 8 -> 誤差: 0.077823 | 判定: ❌ 異常(Anomaly)
ラベル 9 -> 誤差: 0.069749 | 判定: ❌ 異常(Anomaly)
